In [4]:
import pandas as pd

cruzbuy = pd.read_csv('./data/clean/cruzbuy_clean.csv')
cruzbuy.head(5)

,Transaction Date,Merchant Name,Item Description,Category,Subcategory,Item Name,Quantity,Subtotal,Total Price
0,2025-01-02,Complete Book & Media Supply Inc,Quanta And Fields: The Biggest Ideas In The Un...,Published Products,Printed Media,Nan,1.0,$17.68,$17.68
1,2025-01-02,New England Biolabs Inc,Nebridge® Ligase Master Mix,Nan,Nan,Nan,1.0,$55.80,$55.80
2,2025-01-02,New England Biolabs Inc,Nebuilder® Hifi Dna Assembly Master Mix,Nan,Nan,Nan,1.0,"$1,608.00","$1,608.00"
3,2025-01-02,New England Biolabs Inc,Nhei-Hf®,Nan,Nan,Nan,1.0,$46.80,$46.80
4,2025-01-02,New England Biolabs Inc,Noti-Hf®,Nan,Nan,Nan,1.0,$49.80,$49.80


In [6]:
import pandas as pd

# Load cleaned Amazon data
amazon_path = "./data/clean/amazon_clean.csv"
amazon_df = pd.read_csv(amazon_path)

# Parse date + money columns
amazon_df["Transaction Date"] = pd.to_datetime(amazon_df["Transaction Date"], errors="coerce")
amazon_df["Total Price"] = (
    amazon_df["Total Price"]
    .astype(str)
    .str.replace(r"[\$,]", "", regex=True)
    .str.strip()
)
amazon_df["Total Price"] = pd.to_numeric(amazon_df["Total Price"], errors="coerce")

# Drop bad rows
amazon_df = amazon_df.dropna(subset=["Transaction Date", "Total Price"])

# Monthly spend (YYYY-MM)
amazon_monthly_spend = (
    amazon_df
    .assign(Month=amazon_df["Transaction Date"].dt.strftime("%Y-%m"))
    .groupby("Month", as_index=False)["Total Price"]
    .sum()
    .rename(columns={"Total Price": "Amazon Spend"})
)

# Round for readability
amazon_monthly_spend["Amazon Spend"] = amazon_monthly_spend["Amazon Spend"].round(2)

amazon_monthly_spend


,Month,Amazon Spend
0,2025-01,346572.66
1,2025-02,359988.71
2,2025-03,276227.97
3,2025-04,606004.32
4,2025-05,771313.08
5,2025-06,468447.59
6,2025-07,184766.11
7,2025-08,246647.13
8,2025-09,228045.70


In [7]:
# Top 20 purchases by frequency and by total cost
import pandas as pd

dataset_path = './data/clean/cruzbuy_clean.csv'  # swap to amazon_clean.csv, onecard_clean.csv, or bookstore_clean.csv as needed
item_col = 'Item Description'
cost_col = 'Total Price'

df = pd.read_csv(dataset_path).copy()
df[item_col] = df[item_col].fillna('').astype(str).str.strip()
df = df[df[item_col] != '']

df[cost_col] = (
    df[cost_col]
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .str.strip()
)
df[cost_col] = pd.to_numeric(df[cost_col], errors='coerce').fillna(0.0)

top_20_by_frequency = (
    df.groupby(item_col)
    .size()
    .reset_index(name='Purchase Count')
    .sort_values('Purchase Count', ascending=False)
    .head(20)
)

top_20_by_cost = (
    df.groupby(item_col, as_index=False)[cost_col]
    .sum()
    .rename(columns={cost_col: 'Total Cost'})
    .sort_values('Total Cost', ascending=False)
    .head(20)
)

print('Top 20 Most Frequent Purchases')
display(top_20_by_frequency)

print('Top 20 Purchases by Total Cost')
display(top_20_by_cost)


Top 20 Most Frequent Purchases


,Item Description,Purchase Count
9511,Custom Dna Tube,602
26150,Shipping,268
13614,Freight,197
22570,Placeholder - Do Not Close,186
14518,Gratuity,170
8303,Co2 Dry Ice Nuggets,148
20926,Nitrogen Ind Liq 230Lt 22Psi,114
13645,Freight Estimate,104
12011,Ewaste Fees,83
31537,Universal Pipet Tips,61


Top 20 Purchases by Total Cost


,Item Description,Total Cost
3794,Acct#160710 Rachel Carson Dh-Bulk Food Purchase,1500000.00
3793,Acct#160672 College 9/10 Dh - Bulk Food Purchase,1499999.99
3757,Acct# 160678 Cowell Dh - Bulk Food Purchase,1499999.99
17911,Leica Stellaris Wll Confocal Microscope With S...,807280.17
3761,Acct# 160684 Crown Dh - Bulk Food Purchase,800000.00
27595,Standard Compute Nodes; Part #'S • S-As-2124Bt...,682458.00
7587,"Catalyst 9166I Ap (W6E, Tri-Band 4X4, Xor) W/R...",663628.39
12337,Facsdiscover S8 Brvyguv 5Laser,619250.00
7588,"Catalyst 9300 48-Port, 8Xmgig+40X5G 90W Upoe+,...",605836.16
19030,Merrill Market-Bulk Food Purchase,570000.00


In [9]:
# Find raw Amazon rows for specific gift-card product categories
import pandas as pd

amazon_raw_path = './data/raw/amazon.csv'
amazon_raw_df = pd.read_csv(amazon_raw_path).copy()

category_col = 'Amazon-Internal Product Category'
target_categories = [
    'Electronic Gift Card',
    'Gift Card',
    'Target Gift Card',
    'ACD Gift Card',
]

amazon_raw_df[category_col] = amazon_raw_df[category_col].fillna('').astype(str).str.strip()
gift_card_rows = amazon_raw_df[amazon_raw_df[category_col].isin(target_categories)].copy()

print(f'Found {len(gift_card_rows)} raw Amazon rows in the selected gift-card categories.')
display(gift_card_rows[[category_col]].value_counts().reset_index(name='Row Count'))

gift_card_rows.head(20)


Found 515 raw Amazon rows in the selected gift-card categories.


/var/folders/gh/q87xz5kn6tn4w9zcnvxk2jb00000gn/T/ipykernel_32967/2956149287.py:5: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  amazon_raw_df = pd.read_csv(amazon_raw_path).copy()


,Amazon-Internal Product Category,Row Count
0,Electronic Gift Card,459
1,Gift Card,33
2,Target Gift Card,14
3,ACD Gift Card,9


,Order Date,Order ID,Account Group,PO Number,Order Quantity,Currency,Order Subtotal,Order Shipping & Handling,Order Promotion,Order Tax,...,Department,Cost Center,Project Code,Location,Custom Field 1,Seller Name,Seller Credentials,Seller City,Seller State,Seller ZipCode
1,6/4/2025,111-0725963-0664218,UC Santa Cruz Pcard Group,NaN,18,USD,3600.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,09523-627610,Amazon.com,NaN,Seattle,WA,98109
4,6/26/2025,113-6665869-9925065,UC Santa Cruz Pcard Group,NaN,50,USD,2500.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,Amazon.com,NaN,Seattle,WA,98109
13,3/3/2025,111-3715751-1997006,UC Santa Cruz Pcard Group,NaN,7,USD,2565.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,20360-680230,Amazon.com,NaN,Seattle,WA,98109
17,1/9/2025,113-3471214-6591466,UC Santa Cruz Pcard Group,NaN,75,USD,1875.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,Amazon.com,NaN,Seattle,WA,98109
18,3/31/2025,113-2258327-1668241,UC Santa Cruz Pcard Group,NaN,75,USD,1875.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,Amazon.com,NaN,Seattle,WA,98109
25,4/9/2025,111-4407903-3235441,UC Santa Cruz Pcard Group,NaN,24,USD,1764.15,0.0,0.00,17.06,...,NaN,NaN,NaN,NaN,SlugCents Office Supplies,"Amazon Payments, Inc.",NaN,Seattle,WA,98109
30,6/25/2025,113-6903160-3804240,UC Santa Cruz Pcard Group,NaN,60,USD,2200.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,Amazon.com,NaN,Seattle,WA,98109
38,5/29/2025,111-5710924-2067414,UC Santa Cruz Pcard Group,NaN,43,USD,1752.71,0.0,-17.17,35.63,...,NaN,NaN,NaN,NaN,NaN,"Amazon Payments, Inc.",NaN,Seattle,WA,98109
41,3/4/2025,112-5492347-4762638,UC Santa Cruz Pcard Group,NaN,17,USD,1275.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,Room Showcase participants,Amazon.com,NaN,Seattle,WA,98109
43,4/16/2025,112-7375263-3101012,UC Santa Cruz Pcard Group,NaN,12,USD,1271.40,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,"Amazon Payments, Inc.",NaN,Seattle,WA,98109


In [7]:
# List the different categories in each cleaned dataset
import pandas as pd
from pathlib import Path

clean_data_dir = Path('./data/clean')
datasets = {
    'CruzBuy': clean_data_dir / 'cruzbuy_clean.csv',
    'Amazon': clean_data_dir / 'amazon_clean.csv',
    'OneCard': clean_data_dir / 'onecard_clean.csv',
    'Bookstore': clean_data_dir / 'bookstore_clean.csv',
    'ProCard': clean_data_dir / 'procard_clean.csv',
}

category_tables = {}

for dataset_name, dataset_path in datasets.items():
    if not dataset_path.exists():
        print(f'Skipping {dataset_name}: {dataset_path} not found')
        continue

    df = pd.read_csv(dataset_path)
    category_cols = [
        col for col in df.columns
        if col in {'Category', 'Subcategory', 'Merchant Type', 'Transaction Type'}
        or 'category' in col.lower()
    ]

    print(f'\n{dataset_name}: {len(df):,} rows')
    for col in category_cols:
        values = df[col].fillna('Missing').astype(str).str.strip()
        values = values.replace({'': 'Missing', 'nan': 'Missing', 'Nan': 'Missing', 'NaN': 'Missing'})
        counts = values.value_counts().reset_index()
        counts.columns = [col, 'Row Count']
        category_tables[(dataset_name, col)] = counts

        print(f'  {col}: {len(counts):,} categories')
        display(counts)



CruzBuy: 46,412 rows
  Category: 55 categories


,Category,Row Count
0,Missing,15607
1,Laboratory And Measuring And Observing And Tes...,5073
2,Laboratory Supplies,2982
3,Manufacturing Components And Supplies,2756
4,Information Technology Broadcasting And Teleco...,2199
5,Chemical Reagents,2129
6,Molecular Biology,2078
7,Electronic Components And Supplies,1175
8,Office Equipment And Accessories And Supplies,1095
9,Cleaning Equipment And Supplies,977


  Subcategory: 258 categories


,Subcategory,Row Count
0,Missing,17935
1,Laboratory And Scientific Equipment,3238
2,Computer Equipment And Accessories,1557
3,Measuring And Observing And Testing Instruments,1113
4,Molecular Biology Kits,939
...,...,...
253,Raw Materials Processing Machinery,1
254,Stampings And Sheet Components,1
255,Sterilization Equipment And Accessories,1
256,Plastic And Chemical Industries,1



Amazon: 16,102 rows
  Category: 60 categories


...

,Category,Row Count
0,Book Stores,9195
1,Miscellaneous And Special,1852
2,Variety Stores,1746
3,Computer Software Stores,1522
4,Business Services - Other,1140
...,...,...
211,Cathay Pacific,1
212,Veterinary Services,1
213,Tax Payments,1
214,General Contractors - Res,1


  Transaction Type: 2 categories


,Transaction Type,Row Count
0,Purchase,31825
1,Refund,1099


  Merchant Type: 2 categories


,Merchant Type,Row Count
0,External,32811
1,Campus,113


In [ ]:
# Build a reusable broad-category mapping with Gemini
import json
import os
import re
import time
from pathlib import Path

import pandas as pd

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(path, override=False):
        path = Path(path)
        if not path.exists():
            return False
        for line in path.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            key = key.strip()
            if override or key not in os.environ:
                os.environ[key] = value.strip().strip('"').strip("'")
        return True

# Notebook is usually run from backend/data_cleaning, but this also works from repo root.
def find_repo_root(start_path=Path.cwd()):
    start_path = Path(start_path).resolve()
    for path in [start_path, *start_path.parents]:
        if (path / '.env').exists() or (path / 'backend').exists():
            return path
    return start_path


repo_root = find_repo_root()
load_dotenv(repo_root / '.env', override=True)

clean_data_dir = Path('./data/clean')
if not clean_data_dir.exists():
    clean_data_dir = repo_root / 'backend' / 'data_cleaning' / 'data' / 'clean'

datasets = {
    'CruzBuy': clean_data_dir / 'cruzbuy_clean.csv',
    'Amazon': clean_data_dir / 'amazon_clean.csv',
    'OneCard': clean_data_dir / 'onecard_clean.csv',
    'Bookstore': clean_data_dir / 'bookstore_clean.csv',
    'ProCard': clean_data_dir / 'procard_clean.csv',
}

# These are AI recommended broad category suggestions. The dashboard should filter on these values.
broad_categories = [
    'Lab & Research',
    'Chemicals & Reagents',
    'Technology & Electronics',
    'Office & Classroom Supplies',
    'Facilities & Maintenance',
    'Food & Dining',
    'Books & Course Materials',
    'Apparel & Personal Items',
    'Travel & Transportation',
    'Services & Subscriptions',
    'Furniture & Fixtures',
    'Safety & Security',
    'Medical & Health',
    'Gifts & Miscellaneous',
    'Uncategorized',
]

# Start with top-level Category only. We can add subcategories later
category_columns_by_dataset = {
    'CruzBuy': ['Category'],
    'Amazon': ['Category'],
    'OneCard': ['Category'],
    'Bookstore': ['Category'],
    'ProCard': ['Category'],
}

mapping_path = clean_data_dir / 'category_mapping.csv'


def normalize_category(value):
    if pd.isna(value):
        return 'Missing'
    value = str(value).strip()
    if value in {'', 'nan', 'Nan', 'NaN', 'None'}:
        return 'Missing'
    return value


def build_original_category_table():
    rows = []
    for dataset_name, dataset_path in datasets.items():
        if not dataset_path.exists():
            print(f'Skipping {dataset_name}: {dataset_path} not found')
            continue

        df = pd.read_csv(dataset_path)
        for category_column in category_columns_by_dataset.get(dataset_name, ['Category']):
            if category_column not in df.columns:
                continue

            counts = df[category_column].map(normalize_category).value_counts().reset_index()
            counts.columns = ['original_category', 'row_count']
            counts.insert(0, 'category_column', category_column)
            counts.insert(0, 'dataset', dataset_name)
            rows.append(counts)

    if not rows:
        return pd.DataFrame(columns=['dataset', 'category_column', 'original_category', 'row_count'])

    return pd.concat(rows, ignore_index=True)


def strip_json_fences(text):
    text = text.strip()
    text = re.sub(r'^```(?:json)?\\s*', '', text)
    text = re.sub(r'\\s*```$', '', text)
    return text.strip()


def classify_category_batch_with_gemini(batch_df, model_name=None):
    try:
        from google import genai
    except ImportError as exc:
        raise ImportError('Install google-genai first: pip install google-genai') from exc

    # Reload .env each time so a notebook kernel does not keep using an old key.
    load_dotenv(repo_root / '.env', override=True)
    api_key = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
    if not api_key:
        raise ValueError('Missing GEMINI_API_KEY or GOOGLE_API_KEY in your .env file')
    print(f'Using Gemini key ending in ...{api_key[-6:]}')

    client = genai.Client(api_key=api_key)
    model_name = model_name or os.getenv('GEMINI_MODEL', 'gemini-3-flash-preview')

    records = batch_df[['dataset', 'category_column', 'original_category', 'row_count']].to_dict('records')
    prompt = f'''
You are classifying university purchasing transaction categories for a dashboard.

Choose exactly one broad_category from this allowed list:
{json.dumps(broad_categories, indent=2)}

Rules:
- Return JSON only. No markdown.
- Return one object per input record in the same order.
- Use the original category label, dataset name, and row count as context.
- If the label is Missing, Unknown, vague, or impossible to classify, use Uncategorized.
- If it is a gift card, use Gifts & Miscellaneous.
- If it is software, cloud tools, telecom, computers, or electronics, use Technology & Electronics.
- If it is lab equipment, molecular biology, chromatography, spectroscopy, or research supplies, use Lab & Research.
- If it is chemicals, reagents, solvents, gases, or biochemical compounds, use Chemicals & Reagents.

Return this JSON shape:
{{
  "classifications": [
    {{
      "dataset": "...",
      "category_column": "...",
      "original_category": "...",
      "broad_category": "...",
      "confidence": "high|medium|low",
      "notes": "short reason"
    }}
  ]
}}

Input records:
{json.dumps(records, indent=2)}
'''

    response = client.models.generate_content(model=model_name, contents=prompt)
    parsed = json.loads(strip_json_fences(response.text))
    classifications = parsed.get('classifications', [])

    for row in classifications:
        if row.get('broad_category') not in broad_categories:
            row['notes'] = f"Invalid Gemini category '{row.get('broad_category')}'. Needs manual review."
            row['broad_category'] = 'Uncategorized'
            row['confidence'] = 'low'

    return classifications


def generate_category_mapping(batch_size=40, sleep_seconds=0.5, limit=None, overwrite=False):
    original_categories = build_original_category_table()

    if mapping_path.exists() and not overwrite:
        existing_mapping = pd.read_csv(mapping_path)
        key_cols = ['dataset', 'category_column', 'original_category']
        original_categories = original_categories.merge(
            existing_mapping[key_cols + ['broad_category', 'confidence', 'notes']],
            on=key_cols,
            how='left',
        )
        rows_to_classify = original_categories[original_categories['broad_category'].isna()].copy()
    else:
        existing_mapping = None
        original_categories['broad_category'] = pd.NA
        original_categories['confidence'] = pd.NA
        original_categories['notes'] = pd.NA
        rows_to_classify = original_categories.copy()

    rows_to_classify = rows_to_classify.sort_values('row_count', ascending=False)
    if limit is not None:
        rows_to_classify = rows_to_classify.head(limit)

    print(f'{len(rows_to_classify):,} categories need Gemini classification')

    new_rows = []
    for start in range(0, len(rows_to_classify), batch_size):
        batch = rows_to_classify.iloc[start:start + batch_size]
        print(f'Classifying rows {start + 1:,}-{start + len(batch):,} of {len(rows_to_classify):,}')
        new_rows.extend(classify_category_batch_with_gemini(batch))
        time.sleep(sleep_seconds)

    new_mapping = pd.DataFrame(new_rows)
    if new_mapping.empty:
        final_mapping = existing_mapping if existing_mapping is not None else original_categories
    else:
        key_cols = ['dataset', 'category_column', 'original_category']
        final_mapping = original_categories.drop(columns=['broad_category', 'confidence', 'notes']).merge(
            new_mapping[key_cols + ['broad_category', 'confidence', 'notes']],
            on=key_cols,
            how='left',
        )
        if existing_mapping is not None:
            final_mapping = final_mapping.merge(
                existing_mapping[key_cols + ['broad_category', 'confidence', 'notes']],
                on=key_cols,
                how='left',
                suffixes=('', '_existing'),
            )
            for col in ['broad_category', 'confidence', 'notes']:
                final_mapping[col] = final_mapping[col].fillna(final_mapping[f'{col}_existing'])
                final_mapping = final_mapping.drop(columns=[f'{col}_existing'])

    if limit is None:
        final_mapping['broad_category'] = final_mapping['broad_category'].fillna('Uncategorized')
        final_mapping['confidence'] = final_mapping['confidence'].fillna('low')
        final_mapping['notes'] = final_mapping['notes'].fillna('Needs manual review')
    else:
        final_mapping['confidence'] = final_mapping['confidence'].fillna('unclassified')
        final_mapping['notes'] = final_mapping['notes'].fillna('Not classified yet; rerun without limit')
        print('Preview mode: unprocessed categories were left blank, not forced to Uncategorized.')
    final_mapping = final_mapping.sort_values(['dataset', 'category_column', 'row_count'], ascending=[True, True, False])
    final_mapping.to_csv(mapping_path, index=False)
    print(f'Saved mapping to {mapping_path}')
    return final_mapping


def add_broad_category(df, dataset_name, category_column='Category'):
    mapping = pd.read_csv(mapping_path)
    subset = mapping[
        (mapping['dataset'] == dataset_name)
        & (mapping['category_column'] == category_column)
    ][['original_category', 'broad_category']]

    output = df.copy()
    output['_original_category_normalized'] = output[category_column].map(normalize_category)
    output = output.merge(
        subset,
        left_on='_original_category_normalized',
        right_on='original_category',
        how='left',
    )
    output['Broad Category'] = output['broad_category'].fillna('Uncategorized')
    return output.drop(columns=['_original_category_normalized', 'original_category', 'broad_category'])


# First run tip: use a small limit to preview the workflow, then remove limit for the full mapping.
# category_mapping = generate_category_mapping(limit=25)
category_mapping = generate_category_mapping()

# Example after the mapping CSV exists:
# amazon_with_broad_categories = add_broad_category(pd.read_csv(datasets['Amazon']), 'Amazon')
# display(amazon_with_broad_categories[['Category', 'Broad Category']].head())


524 categories need Gemini classification
Classifying rows 1-40 of 524
Using Gemini key ending in ...dsJuls


Classifying rows 41-80 of 524
Using Gemini key ending in ...dsJuls


Classifying rows 81-120 of 524
Using Gemini key ending in ...dsJuls


Classifying rows 121-160 of 524
Using Gemini key ending in ...dsJuls


Classifying rows 161-200 of 524
Using Gemini key ending in ...dsJuls


...

Classifying rows 401-440 of 524
Using Gemini key ending in ...dsJuls


Classifying rows 441-480 of 524
Using Gemini key ending in ...dsJuls


Classifying rows 481-520 of 524
Using Gemini key ending in ...dsJuls


Classifying rows 521-524 of 524
Using Gemini key ending in ...dsJuls


Saved mapping to data/clean/category_mapping.csv
